# Detección de Anomalías en Operaciones de Mercado

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Detectar operaciones anómalas** con análisis estadístico SQL
2. **Calcular z-scores y percentiles** de volumen y precio
3. **Identificar patrones de manipulación** de mercado
4. **Clasificar anomalías** por severidad con `SNOWFLAKE.ML.CLASSIFICATION`
5. **Construir dashboard** de vigilancia de mercado

---

## Lo Que Construirás

Un **sistema de vigilancia de mercado** que detecta anomalías:
- Z-scores de precio y volumen por instrumento
- Detección de operaciones fuera de rango
- Análisis de concentración (operaciones del mismo cliente)
- Dashboard de alertas de mercado

**Valor de Negocio**: Cumplir con MAR (Market Abuse Regulation) y prevenir manipulación.

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.CLASSIFICATION` — Clasificación de anomalías
- `STDDEV()`, `AVG()`, `PERCENTILE_CONT()` — Análisis estadístico
- Window functions — Comparación temporal
- `GENERATOR()` — Datos de mercado sintéticos

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS ANOMALIAS_MERCADO_DB;
CREATE SCHEMA IF NOT EXISTS ANOMALIAS_MERCADO_DB.PUBLIC;
USE SCHEMA ANOMALIAS_MERCADO_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Datos de Operaciones de Mercado

### Qué Vamos a Crear

- **100.000 operaciones** en 20 instrumentos financieros
- **2% de operaciones anómalas** con patrones sospechosos
- Volumen inusual, precio fuera de rango, concentración horaria

In [ ]:
CREATE OR REPLACE TABLE OPERACIONES_MERCADO (
    operacion_id VARCHAR(12) PRIMARY KEY,
    instrumento VARCHAR(20),
    fecha_hora TIMESTAMP,
    tipo VARCHAR(10),
    precio DECIMAL(10,4),
    volumen INTEGER,
    trader_id VARCHAR(10),
    mesa VARCHAR(30),
    es_anomala BOOLEAN
);

INSERT INTO OPERACIONES_MERCADO
WITH instrumentos AS (
    SELECT column1 AS inst, column2 AS precio_base FROM VALUES
    ('IBEX35',9500),('SAN',3.5),('BBVA',6.8),('TEF',4.2),('ITX',32.0),
    ('REP',13.5),('IBE',11.0),('AMS',5.8),('GRF',22.0),('FER',28.5),
    ('BKT',6.5),('SAB',1.2),('CIE',24.0),('ACS',33.0),('ENG',21.0),
    ('MAP',1.9),('COL',7.5),('ELE',11.5),('VIS',8.2),('MRL',9.8)
)
SELECT
    'OP' || LPAD(SEQ4()::STRING, 8, '0'),
    i.inst,
    DATEADD('second', -UNIFORM(0, 2592000, RANDOM()), CURRENT_TIMESTAMP()),
    CASE WHEN UNIFORM(0,1,RANDOM()) < 0.55 THEN 'COMPRA' ELSE 'VENTA' END,
    CASE WHEN UNIFORM(0,100,RANDOM()) < 2
        THEN ROUND(i.precio_base * (1 + UNIFORM(-15, 15, RANDOM()) / 100.0), 4)
        ELSE ROUND(i.precio_base * (1 + UNIFORM(-3, 3, RANDOM()) / 100.0), 4) END,
    CASE WHEN UNIFORM(0,100,RANDOM()) < 2
        THEN UNIFORM(50000, 500000, RANDOM())
        ELSE UNIFORM(100, 10000, RANDOM()) END,
    'TRD' || LPAD(UNIFORM(1, 50, RANDOM())::STRING, 3, '0'),
    ARRAY_CONSTRUCT('Renta Variable','Renta Fija','Derivados','FX','Estructurados')[UNIFORM(0,4,RANDOM())]::VARCHAR,
    CASE WHEN UNIFORM(0,100,RANDOM()) < 2 THEN TRUE ELSE FALSE END
FROM TABLE(GENERATOR(ROWCOUNT => 100000))
CROSS JOIN instrumentos i
WHERE UNIFORM(0,1,RANDOM()) > 0.95;

SELECT es_anomala, COUNT(*), ROUND(AVG(volumen),0) AS vol_medio
FROM OPERACIONES_MERCADO GROUP BY 1;

---

## Paso 3: Análisis Estadístico de Anomalías

### Z-Scores y Percentiles

Calculamos desviaciones estadísticas por instrumento para identificar outliers.

In [ ]:
CREATE OR REPLACE TABLE FEATURES_ANOMALIAS AS
WITH stats_instrumento AS (
    SELECT instrumento,
        AVG(precio) AS precio_medio, STDDEV(precio) AS precio_std,
        AVG(volumen) AS volumen_medio, STDDEV(volumen) AS volumen_std,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY volumen) AS p95_volumen
    FROM OPERACIONES_MERCADO GROUP BY 1
)
SELECT
    o.operacion_id,
    o.volumen::FLOAT,
    o.precio::FLOAT,
    CASE WHEN s.precio_std > 0 THEN ((o.precio - s.precio_medio) / s.precio_std)::FLOAT ELSE 0 END AS zscore_precio,
    CASE WHEN s.volumen_std > 0 THEN ((o.volumen - s.volumen_medio) / s.volumen_std)::FLOAT ELSE 0 END AS zscore_volumen,
    CASE WHEN o.volumen > s.p95_volumen THEN 1 ELSE 0 END::FLOAT AS sobre_p95,
    HOUR(o.fecha_hora)::FLOAT AS hora,
    CASE WHEN o.tipo = 'COMPRA' THEN 1 ELSE 0 END::FLOAT AS es_compra,
    o.es_anomala
FROM OPERACIONES_MERCADO o
JOIN stats_instrumento s ON o.instrumento = s.instrumento;

CREATE OR REPLACE TABLE TRAIN_ANOM AS SELECT * FROM FEATURES_ANOMALIAS SAMPLE (80);
CREATE OR REPLACE TABLE TEST_ANOM AS SELECT * FROM FEATURES_ANOMALIAS WHERE operacion_id NOT IN (SELECT operacion_id FROM TRAIN_ANOM);

CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_anomalias(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_ANOM'),
    TARGET_COLNAME => 'ES_ANOMALA'
);

In [ ]:
CALL detector_anomalias!SHOW_EVALUATION_METRICS();

---

## Paso 4: Generar Alertas de Mercado

In [ ]:
CREATE OR REPLACE TABLE ALERTAS_MERCADO AS
SELECT
    operacion_id, volumen, precio, zscore_precio, zscore_volumen,
    es_anomala AS anomala_real,
    detector_anomalias!PREDICT(
        OBJECT_CONSTRUCT(
            'VOLUMEN', volumen, 'PRECIO', precio,
            'ZSCORE_PRECIO', zscore_precio, 'ZSCORE_VOLUMEN', zscore_volumen,
            'SOBRE_P95', sobre_p95, 'HORA', hora, 'ES_COMPRA', es_compra
        )
    ) AS pred,
    ROUND(pred['probability']['true']::FLOAT * 100, 1) AS prob_anomalia,
    CASE
        WHEN pred['probability']['true']::FLOAT >= 0.7 THEN 'CRITICA'
        WHEN pred['probability']['true']::FLOAT >= 0.4 THEN 'ALTA'
        ELSE 'NORMAL'
    END AS severidad
FROM TEST_ANOM;

SELECT severidad, COUNT(*) FROM ALERTAS_MERCADO GROUP BY 1 ORDER BY 1;

---

## Paso 5: Dashboard de Vigilancia

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Detección de Anomalías — Mercados')
st.markdown('### Vigilancia de Operaciones')

total = session.sql('SELECT COUNT(*) FROM ALERTAS_MERCADO').collect()[0][0]
criticas = session.sql("SELECT COUNT(*) FROM ALERTAS_MERCADO WHERE severidad='CRITICA'").collect()[0][0]

c1, c2 = st.columns(2)
c1.metric('Total Operaciones', f'{total:,}')
c2.metric('Alertas Críticas', f'{criticas:,}')

st.markdown('---')
df = session.sql('SELECT severidad, COUNT(*) AS n FROM ALERTAS_MERCADO GROUP BY 1').to_pandas()
st.bar_chart(df.set_index('SEVERIDAD'))

st.markdown('---')
st.subheader('Operaciones Sospechosas')
df_sosp = session.sql("SELECT operacion_id, volumen, precio, zscore_precio, zscore_volumen, prob_anomalia, severidad FROM ALERTAS_MERCADO WHERE severidad='CRITICA' ORDER BY prob_anomalia DESC LIMIT 20").to_pandas()
st.dataframe(df_sosp)

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS ANOMALIAS_MERCADO_DB;